# Data Science & Machine Learning — Atelier 2 V2
## Apprentissage supervisé : du problème à la comparaison de modèles

### Objectifs
À la fin de cet atelier, vous devez être capables de :
- formuler un problème supervisé avec `X` et `y` ;
- distinguer régression et classification ;
- séparer train et test avant toute transformation apprenante ;
- construire un préprocesseur avec **imputation + scaling + encodage** ;
- construire un pipeline ;
- comparer une **baseline**, une **régression linéaire** et un **Random Forest** ;
- interpréter R², RMSE et MAE ;
- analyser les prédictions et l'importance des variables ;
- expliquer pourquoi un modèle complexe n'est pas automatiquement meilleur.

> **Fil conducteur : baseline → modèle simple → modèle plus complexe → comparaison → interprétation.**

## ÉTAPE 1 — Charger le dataset brut et poser le problème ML

Nous repartons volontairement du **dataset brut**, et non de la version imputée pendant l'EDA.

Pourquoi ? Parce qu'en Machine Learning, les valeurs d'imputation doivent être apprises uniquement sur le **TRAIN**, puis appliquées au TEST.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np

DATA_PATH = "/content/drive/MyDrive/DATASETS/dataset_energie_batiments_pedagogique.csv"
df = pd.read_csv(DATA_PATH)

df.columns = df.columns.str.strip().str.lower()
df.info()

### 1.1 Formuler le problème

Nous cherchons à prédire :

`energy_consumption_kwh`

C'est une valeur numérique continue : nous sommes donc dans un problème de **régression supervisée**.

- `X` = informations utilisées par le modèle ;
- `y` = cible que le modèle apprend à prédire.

In [ ]:
target = "energy_consumption_kwh"

# En apprentissage supervisé, une observation dont la cible y est inconnue
# ne peut pas servir à entraîner ni à évaluer le modèle.
print("Valeurs manquantes dans la cible :", df[target].isna().sum())

df_ml = df.dropna(subset=[target]).copy()

X = df_ml.drop(columns=[target])
y = df_ml[target]

print("Cible :", target)
print("Observations initiales :", len(df))
print("Observations utilisables :", len(df_ml))
print("X :", X.shape)
print("y :", y.shape)

### 1.1 bis — Pourquoi supprimer les lignes sans cible ?

Il faut distinguer deux situations :

- une valeur manquante dans **X** peut éventuellement être imputée par le préprocesseur ;
- une valeur manquante dans **y** signifie que nous ne connaissons pas la réponse attendue.

En apprentissage supervisé, une ligne sans cible ne peut donc pas servir à apprendre la relation `X → y`, ni à mesurer correctement l'erreur du modèle.

> **On peut estimer une feature manquante ; on ne peut pas superviser l'apprentissage avec une réponse inconnue.**

### 1.2 Retirer les identifiants sans valeur explicative

Un identifiant unique permet de reconnaître une ligne, mais n'explique pas nécessairement le phénomène étudié.

In [ ]:
if "building_id" in X.columns:
    X = X.drop(columns=["building_id"])

X.head()

### Question

1. Pourquoi `energy_consumption_kwh` est-elle une cible de régression ?
2. Quelles variables pourraient raisonnablement aider à la prédire ?
3. Certaines variables pourraient-elles constituer une fuite d'information si elles étaient directement calculées à partir de la cible ?

## ÉTAPE 2 — Séparation TRAIN / TEST

Si nous évaluons un modèle sur les données qui ont servi à l'entraîner, nous mesurons surtout sa capacité à mémoriser.

Le split doit donc être effectué **avant** :
- imputation ;
- scaling ;
- encodage ;
- toute transformation qui apprend des paramètres à partir des données.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)

print("TRAIN :", X_train.shape)
print("TEST  :", X_test.shape)

> Environ 70 % des données servent à l'apprentissage et 30 % à l'évaluation.

Le jeu de test doit rester « invisible » pendant la préparation et l'entraînement.

## ÉTAPE 3 — Construire le préprocesseur

### 3.1 Identifier variables numériques et catégorielles

In [ ]:
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numériques :", numeric_features)
print("Catégorielles :", categorical_features)

### 3.2 Pourquoi imputer dans le pipeline ?

Si nous calculons la médiane sur l'ensemble du dataset avant le split, le jeu de test influence indirectement la préparation.

Nous allons donc utiliser :
- **médiane** pour les variables numériques ;
- **mode** (`most_frequent`) pour les variables catégorielles.

Ces valeurs seront apprises uniquement pendant `fit(X_train, y_train)`.

### 3.3 Scaling et encodage

- `StandardScaler` met les variables numériques sur une échelle comparable.
- `OneHotEncoder` transforme les catégories en variables numériques binaires.

⚠️ Tous les modèles ne sont pas sensibles aux échelles :
- KNN, SVM, régressions régularisées, réseaux de neurones : souvent sensibles ;
- arbres et Random Forest : généralement peu ou pas sensibles au scaling.

Nous conservons ici un préprocesseur générique pour pouvoir comparer plusieurs modèles avec une chaîne homogène.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

preprocessor

### Préprocesseur ≠ pipeline complet

- **Préprocesseur** : transforme les données.
- **Pipeline** : enchaîne préparation + modèle.

`données brutes → préprocesseur → modèle → prédiction`

## ÉTAPE 4 — Construire une BASELINE

Avant de tester un modèle sophistiqué, nous avons besoin d'une référence.

Pour une régression, une baseline très simple consiste à toujours prédire la **moyenne de `y_train`**.

Si un modèle ML ne fait pas mieux, sa complexité n'apporte probablement pas grand-chose.

In [ ]:
from sklearn.dummy import DummyRegressor

baseline = DummyRegressor(strategy="mean")

# La baseline apprend uniquement la moyenne de y_train.
baseline.fit(X_train, y_train)
y_pred_baseline = baseline.predict(X_test)

print("Moyenne de y_train :", y_train.mean())
print("Premières prédictions de la baseline :", y_pred_baseline[:10])

### Question — La baseline a-t-elle réellement appris les caractéristiques des bâtiments ?

Non. Elle prédit toujours la même valeur : la moyenne observée sur `y_train`.

Son intérêt est précisément de fournir un **niveau de référence minimal** : un modèle plus complexe doit apporter quelque chose de plus que cette stratégie naïve.

## ÉTAPE 5 — Modèle simple : régression linéaire

La régression linéaire constitue une référence simple et interprétable.

Elle cherche une relation de la forme :

`y = b0 + b1*X1 + b2*X2 + ...`

Elle ne capture pas naturellement toutes les relations non linéaires, mais elle permet de vérifier si un modèle simple suffit déjà.

In [ ]:
from sklearn.linear_model import LinearRegression

linear_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LinearRegression())
])

linear_pipeline.fit(X_train, y_train)
y_pred_linear = linear_pipeline.predict(X_test)

## ÉTAPE 6 — Modèle plus complexe : Random Forest

Une Random Forest combine de nombreux arbres de décision.

Elle peut représenter des relations non linéaires et des interactions complexes entre variables.

Mais :

> **plus complexe ≠ automatiquement meilleur**

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=200,
        random_state=42
    ))
])

rf_pipeline.fit(X_train, y_train)
y_pred_rf = rf_pipeline.predict(X_test)

## ÉTAPE 7 — Évaluer et comparer les modèles

Nous utiliserons trois métriques :

### R²
- `1` : prédictions parfaites ;
- `0` : pas mieux que la référence basée sur la moyenne ;
- `< 0` : moins bien que cette référence sur le jeu évalué.

⚠️ `R² = 0,72` ne signifie pas « 72 % de précision ».

### RMSE
Racine de la moyenne des erreurs quadratiques.
- même unité que la cible ;
- pénalise davantage les grosses erreurs ;
- plus faible = meilleur, à contexte comparable.

### MAE
Erreur absolue moyenne.
- très intuitive ;
- même unité que la cible ;
- moins sensible aux grosses erreurs que la RMSE.

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np

def regression_metrics(y_true, y_pred):
    return {
        "R²": r2_score(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred)
    }

results = pd.DataFrame({
    "Baseline moyenne": regression_metrics(y_test, y_pred_baseline),
    "Régression linéaire": regression_metrics(y_test, y_pred_linear),
    "Random Forest": regression_metrics(y_test, y_pred_rf)
}).T

results

### Questions d'interprétation

1. Le Random Forest fait-il mieux que la baseline ?
2. Fait-il mieux que la régression linéaire ?
3. La complexité supplémentaire est-elle justifiée ?
4. Les écarts sont-ils importants ou faibles ?
5. Les erreurs sont-elles acceptables au regard du contexte métier ?

> Un modèle n'est pas intéressant simplement parce qu'il produit une prédiction.

## ÉTAPE 8 — Analyse visuelle des prédictions

In [ ]:
import matplotlib.pyplot as plt

predictions = {
    "Baseline": y_pred_baseline,
    "Régression linéaire": y_pred_linear,
    "Random Forest": y_pred_rf
}

for name, pred in predictions.items():
    plt.figure(figsize=(6, 6))
    plt.scatter(y_test, pred, alpha=0.6)

    mini = min(y_test.min(), pred.min())
    maxi = max(y_test.max(), pred.max())

    # Diagonale idéale : prédiction = valeur réelle
    plt.plot([mini, maxi], [mini, maxi], linestyle="--")
    plt.xlabel("Valeurs réelles")
    plt.ylabel("Valeurs prédites")
    plt.title(f"Réel vs prédit — {name}")
    plt.show()

### À observer

Un modèle parfait suivrait la diagonale.

Demandez-vous :
- les prédictions sont-elles écrasées autour d'une valeur moyenne ?
- les grandes valeurs sont-elles sous-estimées ?
- les petites valeurs sont-elles surestimées ?
- observe-t-on une structure particulière des erreurs ?

## ÉTAPE 9 — Interpréter le Random Forest : feature importance

In [ ]:
feature_names = rf_pipeline.named_steps["preprocessing"].get_feature_names_out()
importances = rf_pipeline.named_steps["model"].feature_importances_

importance_df = (
    pd.DataFrame({
        "feature": feature_names,
        "importance": importances
    })
    .sort_values("importance", ascending=False)
)

importance_df.head(15)

### Attention : importance ≠ causalité

Si `surface_m2` apparaît comme une variable importante, cela signifie essentiellement :

> « Le modèle s'est beaucoup appuyé sur cette variable pour produire ses prédictions. »

Cela ne prouve pas :

> « La surface cause à elle seule la consommation énergétique. »

Une importance de variable est un outil d'interprétation du modèle, pas une preuve de causalité.

## ÉTAPE 10 — Diagnostic : pourquoi un modèle peut-il être mauvais ?

Un pipeline techniquement correct peut produire un modèle médiocre.

Causes possibles :
- dataset trop petit ;
- variables explicatives insuffisantes ;
- bruit important ;
- relations difficiles à apprendre ;
- cible mal choisie ;
- données peu représentatives ;
- modèle inadapté ;
- hyperparamètres perfectibles.

Exemples de variables qui pourraient manquer dans notre cas :
- nombre d'occupants ;
- usage réel du bâtiment ;
- surface réellement chauffée ;
- type de chauffage ;
- niveau d'isolation ;
- conditions climatiques ;
- durée d'occupation.

> **Un mauvais score n'est pas forcément un échec : il peut révéler les limites des données.**

## ÉTAPE 11 — Extension courte : découvrir la classification

Jusqu'ici, `y` était une valeur numérique : **régression**.

Si `y` est une catégorie, nous faisons de la **classification**.

Si le dataset contient `energy_class`, nous pouvons réaliser une courte démonstration.

⚠️ Si `energy_class` est calculée directement à partir de `energy_consumption_kwh`, utiliser la consommation comme variable explicative constituerait une **fuite d'information**. Nous la retirons donc ici.

In [ ]:
if "energy_class" in df.columns:
    target_clf = "energy_class"

    X_clf = df.drop(columns=[target_clf])
    y_clf = df[target_clf]

    # Identifiant inutile
    X_clf = X_clf.drop(columns=["building_id"], errors="ignore")

    # Prudence : si la classe énergétique est dérivée de la consommation,
    # on retire la consommation de X pour éviter une cible "donnée" au modèle.
    X_clf = X_clf.drop(columns=["energy_consumption_kwh"], errors="ignore")

    Xc_train, Xc_test, yc_train, yc_test = train_test_split(
        X_clf,
        y_clf,
        test_size=0.30,
        random_state=42,
        stratify=y_clf if y_clf.value_counts().min() >= 2 else None
    )

    num_clf = Xc_train.select_dtypes(include=["number"]).columns.tolist()
    cat_clf = Xc_train.select_dtypes(include=["object", "category"]).columns.tolist()

    preprocessor_clf = ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_clf),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_clf)
    ])

    from sklearn.ensemble import RandomForestClassifier

    clf_pipeline = Pipeline([
        ("preprocessing", preprocessor_clf),
        ("model", RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            class_weight="balanced"
        ))
    ])

    clf_pipeline.fit(Xc_train, yc_train)
    yc_pred = clf_pipeline.predict(Xc_test)

    print("Classification réalisée.")
else:
    print("La colonne 'energy_class' n'existe pas dans ce dataset : extension à adapter.")

### Évaluer une classification

In [ ]:
if "energy_class" in df.columns:
    from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

    print("Accuracy :", accuracy_score(yc_test, yc_pred))
    print()
    print(classification_report(yc_test, yc_pred, zero_division=0))

    ConfusionMatrixDisplay.from_predictions(yc_test, yc_pred)
    plt.title("Matrice de confusion")
    plt.show()

### À retenir sur la classification

Les métriques ne sont plus exactement les mêmes que pour une régression :
- **accuracy** : proportion totale de bonnes classifications ;
- **precision** : parmi les prédictions d'une classe, combien sont correctes ?
- **recall** : parmi les vrais éléments d'une classe, combien ont été retrouvés ?
- **F1-score** : compromis entre precision et recall ;
- **matrice de confusion** : quelles classes sont confondues ?

Nous approfondirons surtout ici la logique générale : **même démarche supervisée, mais cible et métriques différentes.**

## ÉTAPE 12 — Synthèse de l'Atelier 2

### Rappel important sur les valeurs manquantes

- **NaN dans X** : peut être traité dans le préprocesseur (imputation apprise sur le TRAIN).
- **NaN dans y** : l'observation est retirée du problème supervisé, car la réponse attendue est inconnue.

### La chaîne complète

`Question métier`
→ `X / y`
→ `Train / Test`
→ `Prétraitement`
→ `Baseline`
→ `Modèle simple`
→ `Modèle plus complexe`
→ `Prédiction`
→ `Évaluation`
→ `Interprétation`

### Questions auxquelles vous devez maintenant savoir répondre

1. Pourquoi séparer train et test ?
2. Pourquoi l'imputation doit-elle être apprise sur le train ?
3. Quelle différence entre préprocesseur et pipeline ?
4. Quelle différence entre régression et classification ?
5. Pourquoi construire une baseline ?
6. Que signifie un R² négatif ?
7. Quelle différence entre RMSE et MAE ?
8. Pourquoi un Random Forest n'est-il pas automatiquement préférable à une régression linéaire ?
9. Que signifie une feature importance ?
10. Pourquoi importance ne signifie-t-elle pas causalité ?

### Transition

> Un modèle qui obtient un bon score est-il nécessairement un bon modèle ?

Non : il faut encore étudier la **généralisation**, l'**underfitting**, l'**overfitting**, le **biais-variance**, la **data leakage** et la qualité du signal disponible.